# 06 — Ensembles Combinados: Voting e Stacking

Este notebook implementa as **estratégias de combinação avançada** do TCC:

| Método | Como combina | Aprende nova camada? |
|---|---|---|
| **Voting** | Combina predições dos base models diretamente | ❌ Não |
| **Stacking** | Treina um meta-modelo sobre as predições dos base models | ✅ Sim |

## Base Models selecionados (todos × TF-IDF)

| Modelo | F1-Score (CV) | Justificativa |
|---|:---:|---|
| SVM | 0.8933 | Melhor geral — fronteira de margem máxima |
| Random Forest | 0.8784 | Melhor ensemble — bagging de árvores |
| Naive Bayes | 0.8768 | Modelo complementar — assume independência |
| XGBoost | 0.8668 | Boosting com regularização L1/L2 |
| Gradient Boosting | 0.8657 | Boosting por gradiente clássico |

## Combinações testadas:
1. `SVM + NB + RF` — top 3 modelos de famílias distintas
2. `SVM + NB + RF + XGBoost` — adiciona boosting
3. `SVM + NB + RF + XGBoost + GB` — combinação ampla

> **Pré-requisito:** Execute os Notebooks 04 e 05 para gerar os modelos `.joblib` em `results/metrics/`.

## 1. Importações e Carregamento

In [1]:
import sys
sys.path.append('..')

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

from src.models import salvar_modelo

os.makedirs('../results/metrics', exist_ok=True)

# Fold estratificado compartilhado — random_state=42 garante folds idênticos
# para comparação justa entre Voting e Stacking
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('✅ Importações concluídas!')

✅ Importações concluídas!


In [2]:
# Carrega apenas o split TF-IDF — todos os base models usam esta representação.
X_train_tfidf, X_test_tfidf, y_train, y_test = joblib.load('../data/processed/splits_tfidf.joblib')

print(f'TF-IDF — Treino: {X_train_tfidf.shape} | Teste: {X_test_tfidf.shape}')
print(f'Classes no treino: {dict(y_train.value_counts())}')

TF-IDF — Treino: (3021, 5000) | Teste: (756, 5000)
Classes no treino: {0: np.int64(2149), 1: np.int64(872)}


In [3]:
# Carrega os melhores modelos já treinados e salvos nos Notebooks 04 e 05.
# IMPORTANTE: no Stacking, o sklearn refaz o fit internamente via CV para
# gerar predições out-of-fold (OOF) sem data leakage. Os hiperparâmetros
# dos modelos carregados são preservados — apenas o treino é repetido.
svm_tfidf = joblib.load('../results/metrics/svm_tfidf_best.joblib')
rf_tfidf  = joblib.load('../results/metrics/rf_tfidf_best.joblib')
nb_tfidf  = joblib.load('../results/metrics/nb_tfidf_best.joblib')
xgb_tfidf = joblib.load('../results/metrics/xgb_tfidf_best.joblib')
gb_tfidf  = joblib.load('../results/metrics/gb_tfidf_best.joblib')

print('✅ Modelos carregados com sucesso!')

✅ Modelos carregados com sucesso!


## 2. Voting Classifier

O **Voting Classifier** combina predições sem aprender nova camada:
- **Hard**: voto majoritário — $\hat{y} = \text{mode}(\hat{y}_1, ..., \hat{y}_N)$
- **Soft**: média das probabilidades — $\hat{y} = \arg\max_k \frac{1}{N} \sum p_i(k)$

### 2.1 Voting Hard — SVM + NB + RF

In [4]:
print('=' * 55)
print('  Hard Voting: SVM + NB + RF')
print('=' * 55)

v_hard_1 = VotingClassifier(
    estimators=[('svm', svm_tfidf), ('nb', nb_tfidf), ('rf', rf_tfidf)],
    voting='hard', n_jobs=-1,
)
scores_vh1 = cross_val_score(v_hard_1, X_train_tfidf, y_train,
                             cv=cv, scoring='f1_weighted', n_jobs=-1)
print(f'F1-Score (5-Fold CV): {scores_vh1.mean():.4f} ± {scores_vh1.std():.4f}')
v_hard_1.fit(X_train_tfidf, y_train)
salvar_modelo(v_hard_1, '../results/metrics/voting_hard_svm_nb_rf.joblib')

  Hard Voting: SVM + NB + RF
F1-Score (5-Fold CV): 0.8975 ± 0.0138
Modelo salvo em: ../results/metrics/voting_hard_svm_nb_rf.joblib


### 2.2 Voting Soft — SVM + NB + RF

In [5]:
print('=' * 55)
print('  Soft Voting: SVM + NB + RF')
print('=' * 55)

v_soft_1 = VotingClassifier(
    estimators=[('svm', svm_tfidf), ('nb', nb_tfidf), ('rf', rf_tfidf)],
    voting='soft', n_jobs=-1,
)
scores_vs1 = cross_val_score(v_soft_1, X_train_tfidf, y_train,
                             cv=cv, scoring='f1_weighted', n_jobs=-1)
print(f'F1-Score (5-Fold CV): {scores_vs1.mean():.4f} ± {scores_vs1.std():.4f}')
v_soft_1.fit(X_train_tfidf, y_train)
salvar_modelo(v_soft_1, '../results/metrics/voting_soft_svm_nb_rf.joblib')

  Soft Voting: SVM + NB + RF
F1-Score (5-Fold CV): 0.9016 ± 0.0184
Modelo salvo em: ../results/metrics/voting_soft_svm_nb_rf.joblib


### 2.3 Voting Hard — SVM + NB + RF + XGBoost

In [6]:
print('=' * 55)
print('  Hard Voting: SVM + NB + RF + XGBoost')
print('=' * 55)

v_hard_2 = VotingClassifier(
    estimators=[('svm', svm_tfidf), ('nb', nb_tfidf), ('rf', rf_tfidf), ('xgb', xgb_tfidf)],
    voting='hard', n_jobs=-1,
)
scores_vh2 = cross_val_score(v_hard_2, X_train_tfidf, y_train,
                             cv=cv, scoring='f1_weighted', n_jobs=-1)
print(f'F1-Score (5-Fold CV): {scores_vh2.mean():.4f} ± {scores_vh2.std():.4f}')
v_hard_2.fit(X_train_tfidf, y_train)
salvar_modelo(v_hard_2, '../results/metrics/voting_hard_svm_nb_rf_xgb.joblib')

  Hard Voting: SVM + NB + RF + XGBoost
F1-Score (5-Fold CV): 0.8863 ± 0.0120
Modelo salvo em: ../results/metrics/voting_hard_svm_nb_rf_xgb.joblib


### 2.4 Voting Soft — SVM + NB + RF + XGBoost

In [7]:
print('=' * 55)
print('  Soft Voting: SVM + NB + RF + XGBoost')
print('=' * 55)

v_soft_2 = VotingClassifier(
    estimators=[('svm', svm_tfidf), ('nb', nb_tfidf), ('rf', rf_tfidf), ('xgb', xgb_tfidf)],
    voting='soft', n_jobs=-1,
)
scores_vs2 = cross_val_score(v_soft_2, X_train_tfidf, y_train,
                             cv=cv, scoring='f1_weighted', n_jobs=-1)
print(f'F1-Score (5-Fold CV): {scores_vs2.mean():.4f} ± {scores_vs2.std():.4f}')
v_soft_2.fit(X_train_tfidf, y_train)
salvar_modelo(v_soft_2, '../results/metrics/voting_soft_svm_nb_rf_xgb.joblib')

  Soft Voting: SVM + NB + RF + XGBoost
F1-Score (5-Fold CV): 0.8994 ± 0.0144
Modelo salvo em: ../results/metrics/voting_soft_svm_nb_rf_xgb.joblib


### 2.5 Voting Hard — SVM + NB + RF + XGBoost + GB

In [8]:
print('=' * 55)
print('  Hard Voting: SVM + NB + RF + XGBoost + GB')
print('=' * 55)

v_hard_3 = VotingClassifier(
    estimators=[('svm', svm_tfidf), ('nb', nb_tfidf), ('rf', rf_tfidf),
                ('xgb', xgb_tfidf), ('gb', gb_tfidf)],
    voting='hard', n_jobs=-1,
)
scores_vh3 = cross_val_score(v_hard_3, X_train_tfidf, y_train,
                             cv=cv, scoring='f1_weighted', n_jobs=-1)
print(f'F1-Score (5-Fold CV): {scores_vh3.mean():.4f} ± {scores_vh3.std():.4f}')
v_hard_3.fit(X_train_tfidf, y_train)
salvar_modelo(v_hard_3, '../results/metrics/voting_hard_svm_nb_rf_xgb_gb.joblib')

  Hard Voting: SVM + NB + RF + XGBoost + GB
F1-Score (5-Fold CV): 0.8886 ± 0.0108
Modelo salvo em: ../results/metrics/voting_hard_svm_nb_rf_xgb_gb.joblib


### 2.6 Voting Soft — SVM + NB + RF + XGBoost + GB

In [9]:
print('=' * 55)
print('  Soft Voting: SVM + NB + RF + XGBoost + GB')
print('=' * 55)

v_soft_3 = VotingClassifier(
    estimators=[('svm', svm_tfidf), ('nb', nb_tfidf), ('rf', rf_tfidf),
                ('xgb', xgb_tfidf), ('gb', gb_tfidf)],
    voting='soft', n_jobs=-1,
)
scores_vs3 = cross_val_score(v_soft_3, X_train_tfidf, y_train,
                             cv=cv, scoring='f1_weighted', n_jobs=-1)
print(f'F1-Score (5-Fold CV): {scores_vs3.mean():.4f} ± {scores_vs3.std():.4f}')
v_soft_3.fit(X_train_tfidf, y_train)
salvar_modelo(v_soft_3, '../results/metrics/voting_soft_svm_nb_rf_xgb_gb.joblib')

  Soft Voting: SVM + NB + RF + XGBoost + GB
F1-Score (5-Fold CV): 0.8940 ± 0.0153
Modelo salvo em: ../results/metrics/voting_soft_svm_nb_rf_xgb_gb.joblib


## 3. Resumo — Voting Classifier (Hard vs Soft)

In [10]:
ref_svm = 0.8933

resultados_voting = pd.DataFrame([
    {'Método': 'Voting Hard', 'Combinação': 'SVM + NB + RF',               'F1-Score (CV)': round(scores_vh1.mean(), 4), 'Std': round(scores_vh1.std(), 4)},
    {'Método': 'Voting Soft', 'Combinação': 'SVM + NB + RF',               'F1-Score (CV)': round(scores_vs1.mean(), 4), 'Std': round(scores_vs1.std(), 4)},
    {'Método': 'Voting Hard', 'Combinação': 'SVM + NB + RF + XGBoost',     'F1-Score (CV)': round(scores_vh2.mean(), 4), 'Std': round(scores_vh2.std(), 4)},
    {'Método': 'Voting Soft', 'Combinação': 'SVM + NB + RF + XGBoost',     'F1-Score (CV)': round(scores_vs2.mean(), 4), 'Std': round(scores_vs2.std(), 4)},
    {'Método': 'Voting Hard', 'Combinação': 'SVM + NB + RF + XGBoost + GB', 'F1-Score (CV)': round(scores_vh3.mean(), 4), 'Std': round(scores_vh3.std(), 4)},
    {'Método': 'Voting Soft', 'Combinação': 'SVM + NB + RF + XGBoost + GB', 'F1-Score (CV)': round(scores_vs3.mean(), 4), 'Std': round(scores_vs3.std(), 4)},
])

resultados_voting = resultados_voting.sort_values('F1-Score (CV)', ascending=False).reset_index(drop=True)
display(resultados_voting)

melhor = resultados_voting.iloc[0]
print(f'\nReferência (SVM × TF-IDF): {ref_svm}')
print(f"Melhor Voting: {melhor['Método']} | {melhor['Combinação']} → {melhor['F1-Score (CV)']}")
print(f'Δ vs SVM: {melhor["F1-Score (CV)"] - ref_svm:+.4f}')

resultados_voting.to_csv('../results/metrics/resultados_voting.csv', index=False)
print('✅ resultados_voting.csv salvo em results/metrics/')

,Método,Combinação,F1-Score (CV),Std
0,Voting Soft,SVM + NB + RF,0.9016,0.0184
1,Voting Soft,SVM + NB + RF + XGBoost,0.8994,0.0144
2,Voting Hard,SVM + NB + RF,0.8975,0.0138
3,Voting Soft,SVM + NB + RF + XGBoost + GB,0.8940,0.0153
4,Voting Hard,SVM + NB + RF + XGBoost + GB,0.8886,0.0108
5,Voting Hard,SVM + NB + RF + XGBoost,0.8863,0.0120



Referência (SVM × TF-IDF): 0.8933
Melhor Voting: Voting Soft | SVM + NB + RF → 0.9016
Δ vs SVM: +0.0083
✅ resultados_voting.csv salvo em results/metrics/


## 4. Stacking Classifier

O **Stacking** treina um **meta-modelo** para aprender a combinar as predições dos base models:

```
                    ┌─ SVM(x)  ─┐
X (TF-IDF) ─────── ├─ NB(x)   ─┤ ──► [p_svm, p_nb, p_rf, ...] ──► Meta-Modelo ──► ŷ
                    ├─ RF(x)   ─┤              (Logistic Regression)
                    └─ XGB(x)  ─┘
                       Nível 0                  Nível 1
```

**Como evitar data leakage?** O sklearn usa **5-Fold CV internamente**: cada fold gera predições *out-of-fold* (OOF) dos base models, que são usadas para treinar o meta-modelo — evitando que ele veja predições de exemplos que os base models já viram.

**Meta-modelo: Logistic Regression**
- Recebe apenas **3–5 colunas** (probabilidades dos base models) — modelo linear é ideal
- Coeficientes interpretáveis: mostram o peso atribuído a cada base model
- Regularização L2 embutida evita overfitting

### 4.1 Stacking — SVM + NB + RF → Logistic Regression

In [11]:
print('=' * 55)
print('  Stacking: SVM + NB + RF → LogReg')
print('=' * 55)

stacking_1 = StackingClassifier(
    estimators=[
        ('svm', svm_tfidf),
        ('nb',  nb_tfidf),
        ('rf',  rf_tfidf),
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5,            # Folds internos para gerar OOF predições sem data leakage
    passthrough=False,
    n_jobs=-1,
)

scores_s1 = cross_val_score(stacking_1, X_train_tfidf, y_train,
                            cv=cv, scoring='f1_weighted', n_jobs=-1)
print(f'F1-Score (5-Fold CV): {scores_s1.mean():.4f} ± {scores_s1.std():.4f}')
stacking_1.fit(X_train_tfidf, y_train)
salvar_modelo(stacking_1, '../results/metrics/stacking_svm_nb_rf.joblib')

  Stacking: SVM + NB + RF → LogReg
F1-Score (5-Fold CV): 0.9006 ± 0.0189
Modelo salvo em: ../results/metrics/stacking_svm_nb_rf.joblib


### 4.2 Stacking — SVM + NB + RF + XGBoost → Logistic Regression

In [12]:
print('=' * 55)
print('  Stacking: SVM + NB + RF + XGBoost → LogReg')
print('=' * 55)

stacking_2 = StackingClassifier(
    estimators=[
        ('svm', svm_tfidf),
        ('nb',  nb_tfidf),
        ('rf',  rf_tfidf),
        ('xgb', xgb_tfidf),
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5,
    passthrough=False,
    n_jobs=-1,
)

scores_s2 = cross_val_score(stacking_2, X_train_tfidf, y_train,
                            cv=cv, scoring='f1_weighted', n_jobs=-1)
print(f'F1-Score (5-Fold CV): {scores_s2.mean():.4f} ± {scores_s2.std():.4f}')
stacking_2.fit(X_train_tfidf, y_train)
salvar_modelo(stacking_2, '../results/metrics/stacking_svm_nb_rf_xgb.joblib')

  Stacking: SVM + NB + RF + XGBoost → LogReg
F1-Score (5-Fold CV): 0.8993 ± 0.0184
Modelo salvo em: ../results/metrics/stacking_svm_nb_rf_xgb.joblib


### 4.3 Stacking — SVM + NB + RF + XGBoost + GB → Logistic Regression

In [13]:
print('=' * 55)
print('  Stacking: SVM + NB + RF + XGBoost + GB → LogReg')
print('=' * 55)

# Vantagem do Stacking sobre Voting com modelos correlacionados (XGBoost + GB):
# o meta-modelo pode aprender a ignorar um deles, atribuindo peso baixo.
# No Voting todos têm peso igual — o Stacking é adaptativo.
stacking_3 = StackingClassifier(
    estimators=[
        ('svm', svm_tfidf),
        ('nb',  nb_tfidf),
        ('rf',  rf_tfidf),
        ('xgb', xgb_tfidf),
        ('gb',  gb_tfidf),
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5,
    passthrough=False,
    n_jobs=-1,
)

scores_s3 = cross_val_score(stacking_3, X_train_tfidf, y_train,
                            cv=cv, scoring='f1_weighted', n_jobs=-1)
print(f'F1-Score (5-Fold CV): {scores_s3.mean():.4f} ± {scores_s3.std():.4f}')
stacking_3.fit(X_train_tfidf, y_train)
salvar_modelo(stacking_3, '../results/metrics/stacking_svm_nb_rf_xgb_gb.joblib')

  Stacking: SVM + NB + RF + XGBoost + GB → LogReg
F1-Score (5-Fold CV): 0.8992 ± 0.0171
Modelo salvo em: ../results/metrics/stacking_svm_nb_rf_xgb_gb.joblib


## 5. Resumo — Stacking Classifier

In [14]:
ref_svm    = 0.8933
ref_voting = 0.9016  # Melhor Voting: Soft SVM + NB + RF

resultados_stacking = pd.DataFrame([
    {'Método': 'Stacking → LogReg', 'Combinação': 'SVM + NB + RF',               'F1-Score (CV)': round(scores_s1.mean(), 4), 'Std': round(scores_s1.std(), 4)},
    {'Método': 'Stacking → LogReg', 'Combinação': 'SVM + NB + RF + XGBoost',     'F1-Score (CV)': round(scores_s2.mean(), 4), 'Std': round(scores_s2.std(), 4)},
    {'Método': 'Stacking → LogReg', 'Combinação': 'SVM + NB + RF + XGBoost + GB', 'F1-Score (CV)': round(scores_s3.mean(), 4), 'Std': round(scores_s3.std(), 4)},
])

resultados_stacking = resultados_stacking.sort_values('F1-Score (CV)', ascending=False).reset_index(drop=True)
display(resultados_stacking)

melhor_s = resultados_stacking.iloc[0]
print(f'\nReferência SVM × TF-IDF:  {ref_svm}')
print(f'Referência melhor Voting: {ref_voting}')
print(f"Melhor Stacking:          {melhor_s['F1-Score (CV)']}")
print(f'Δ vs SVM:     {melhor_s["F1-Score (CV)"] - ref_svm:+.4f}')
print(f'Δ vs Voting:  {melhor_s["F1-Score (CV)"] - ref_voting:+.4f}')

resultados_stacking.to_csv('../results/metrics/resultados_stacking.csv', index=False)
print('✅ resultados_stacking.csv salvo em results/metrics/')

,Método,Combinação,F1-Score (CV),Std
0,Stacking → LogReg,SVM + NB + RF,0.9006,0.0189
1,Stacking → LogReg,SVM + NB + RF + XGBoost,0.8993,0.0184
2,Stacking → LogReg,SVM + NB + RF + XGBoost + GB,0.8992,0.0171



Referência SVM × TF-IDF:  0.8933
Referência melhor Voting: 0.9016
Melhor Stacking:          0.9006
Δ vs SVM:     +0.0073
Δ vs Voting:  -0.0010
✅ resultados_stacking.csv salvo em results/metrics/


## 6. Resumo Geral — Voting vs Stacking

In [15]:
# Lê os CSVs individuais — funciona mesmo sem reexecutar as células acima
df_v = pd.read_csv('../results/metrics/resultados_voting.csv')
df_s = pd.read_csv('../results/metrics/resultados_stacking.csv')

# Ambos os CSVs já têm coluna 'Método' padronizada:
# Voting: 'Voting Hard' / 'Voting Soft'
# Stacking: 'Stacking → LogReg'
colunas = ['Método', 'Combinação', 'F1-Score (CV)', 'Std']
resumo_geral = pd.concat([df_v[colunas], df_s[colunas]], ignore_index=True)
resumo_geral = resumo_geral.sort_values('F1-Score (CV)', ascending=False).reset_index(drop=True)

display(resumo_geral)

resumo_geral.to_csv('../results/metrics/resultados_combinados.csv', index=False)
print('✅ resultados_combinados.csv salvo em results/metrics/')

melhor = resumo_geral.iloc[0]
print(f'\n🏆 Melhor configuração geral do Notebook 06:')
print(f"   Método    : {melhor['Método']}")
print(f"   Combinação: {melhor['Combinação']}")
print(f"   F1 (CV)   : {melhor['F1-Score (CV)']}")

,Método,Combinação,F1-Score (CV),Std
0,Voting Soft,SVM + NB + RF,0.9016,0.0184
1,Stacking → LogReg,SVM + NB + RF,0.9006,0.0189
2,Voting Soft,SVM + NB + RF + XGBoost,0.8994,0.0144
3,Stacking → LogReg,SVM + NB + RF + XGBoost,0.8993,0.0184
4,Stacking → LogReg,SVM + NB + RF + XGBoost + GB,0.8992,0.0171
5,Voting Hard,SVM + NB + RF,0.8975,0.0138
6,Voting Soft,SVM + NB + RF + XGBoost + GB,0.8940,0.0153
7,Voting Hard,SVM + NB + RF + XGBoost + GB,0.8886,0.0108
8,Voting Hard,SVM + NB + RF + XGBoost,0.8863,0.0120


✅ resultados_combinados.csv salvo em results/metrics/

🏆 Melhor configuração geral do Notebook 06:
   Método    : Voting Soft
   Combinação: SVM + NB + RF
   F1 (CV)   : 0.9016
